<div style="color:#3c4d5a; border-top: 7px solid #42A5F5; border-bottom: 7px solid #42A5F5; padding: 5px; text-align: center; text-transform: uppercase"><h1>Aplicaciones de las Redes Bayesianas como herramientas de soporte a la toma de decisiones.</h1> </div> 

Desarrollado por: Domenica Beltran y Mario Reinoso

- [Enunciado de la práctica](#enunciado)

- [1. Cargar el dataset](#dataset)

- [2. Fase de Preparación](#prep)

- [3. Fase de Modelado](model)

- [4. Fase de Predicción de Nuevos Samples](#predict)

- [Conclusiones](#conclusiones)

- [Referencias](#referencias)

## Introducción

La presente práctica aborda la implementación y evaluación del algoritmo probabilístico Naive Bayes para la clasificación y diagnóstico automatizado de perfiles clínicos relacionados con la diabetes. En el ámbito biomédico, el procesamiento de datos heterogéneos que combinan variables cualitativas y cuantitativas representa un desafío crítico para el desarrollo de sistemas de soporte a la decisión médica (Kuhn & Johnson, 2013). A diferencia de los enfoques geométricos basados en distancias analíticas, Naive Bayes fundamenta su operación en el teorema de Bayes y en el principio de verosimilitud, calculando la probabilidad posterior de una condición patológica a partir de las frecuencias observadas en una muestra histórica (James et al., 2013). La importancia de esta práctica radica en estructurar un flujo de trabajo metodológico que contemple desde la codificación adaptada y discretización de variables clínicas de pacientes, hasta la mitigación de probabilidades nulas mediante técnicas de suavizado estadístico, permitiendo analizar la viabilidad de los modelos probabilísticos frente a la naturaleza multivariante de los datos de salud (Hastie et al., 2009).

<div id="enunciado" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Enunciado de la práctica</h2> </div>

Se dispone de un dataset en formato CSV con información básica de pacientes ecuatorianos, incluyendo variables demográficas y de salud. El objetivo es aplicar un flujo básico de Machine Learning utilizando un modelo Naive Bayes para predecir si un nuevo paciente tiene o no diabetes.

Dataset:<br>

sexo,ciudad,colesterol,edad,diabetes <br>
1,Cuenca,bajo,18,no <br>
2,Quito,alto,52,si <br>
2,Guayaquil,medio,34,no <br>
1,Loja,alto,61,si <br>
2,Ambato,medio,45,no <br>
1,Machala,muy alto,67,si <br>

- Fase de Preparación: Realizar preparación de datos, 
- Fase de Modelado: Desarrollar un Modelo Naive Bayes
- Fase de Predicción de Nuevos Samples: Realizar la predicción con al menos dos nuevos samples (ejemplos) con el modelo Naive Bayes.
- Presentar conclusiones y referencias en formato APA.

<div id="Importacion" style="color:#106ba3"><h3>Importación de librerías</h3> </div>

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import CategoricalNB
import copy

<div id="dataset" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>1. Cargar el dataset</h2> </div>

### Nombre del dataset: Dataset de Diabetes en Pacientes Ecuatorianos

Enlace: [Disponible en formato CSV local / No especificado]

Número de Variables (o atributos): 5 (sexo, ciudad, colesterol, edad, diabetes)

Número de instancias (pacientes): 6 (en la muestra proporcionada)

Salida: diabetes (NO: no tiene diabetes, SI: tiene diabetes)

In [28]:
nombresVariables = ['sexo', 'ciudad', 'colesterol', 'edad', 'diabetes']
dataset = "data.csv"
dfOriginal = pd.read_csv(dataset, sep=',', names=nombresVariables, header=0)

# Verificación de dimensiones
print('cantidad de observaciones (pacientes): ', dfOriginal.shape[0])
print('cantidad de variables: ', dfOriginal.shape[1])
print('Dimensiones:', dfOriginal.shape)

# Visualización del estado inicial
dfOriginal.head(6)

cantidad de observaciones (pacientes):  6
cantidad de variables:  5
Dimensiones: (6, 5)


,sexo,ciudad,colesterol,edad,diabetes
0,1,Cuenca,bajo,18,no
1,2,Quito,alto,52,si
2,2,Guayaquil,medio,34,no
3,1,Loja,alto,61,si
4,2,Ambato,medio,45,no
5,1,Machala,muy alto,67,si


<div id="prep" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>2. Fase de Preparación</h2> </div>

In [29]:
# Creación de una copia para la preparación de datos
dfPreparado = dfOriginal.copy()

# 1. Codificación de Variable Nominal ('ciudad') usando LabelEncoder
encoder_ciudad = LabelEncoder()
dfPreparado['ciudad'] = encoder_ciudad.fit_transform(dfPreparado['ciudad'])

# 2. Codificación de Variable Ordinal ('colesterol') respetando la jerarquía interna
mapeo_colesterol = {'bajo': 0, 'medio': 1, 'alto': 2, 'muy alto': 3}
dfPreparado['colesterol'] = dfPreparado['colesterol'].map(mapeo_colesterol)

# Visualización del progreso de codificación
dfPreparado.head(6)

,sexo,ciudad,colesterol,edad,diabetes
0,1,1,0,18,no
1,2,5,2,52,si
2,2,2,1,34,no
3,1,3,2,61,si
4,2,0,1,45,no
5,1,4,3,67,si


In [17]:
# Transformación de la variable continua 'edad' en rangos categóricos numéricos
# Rangos: 0-30 (Joven = 0), 31-50 (Adulto = 1), 51-100 (Mayor = 2)
dfPreparado['edad'] = pd.cut(dfPreparado['edad'], bins=[0, 30, 50, 100], labels=[0, 1, 2]).astype(int)

# Visualización del progreso tras la discretización
dfPreparado.head(6)

,sexo,ciudad,colesterol,edad,diabetes
0,1,1,0,0,no
1,2,5,2,2,si
2,2,2,1,1,no
3,1,3,2,2,si
4,2,0,1,1,no
5,1,4,3,2,si


In [18]:
# Codificación de la variable objetivo 'diabetes' (no = 0, si = 1)
encoder_diabetes = LabelEncoder()
dfPreparado['diabetes'] = encoder_diabetes.fit_transform(dfPreparado['diabetes'])

# Visualización del dataframe final preparado
dfPreparado.head(6)

,sexo,ciudad,colesterol,edad,diabetes
0,1,1,0,0,0
1,2,5,2,2,1
2,2,2,1,1,0
3,1,3,2,2,1
4,2,0,1,1,0
5,1,4,3,2,1


In [19]:
# Separación de variables predictoras (X) y variable dependiente (y)
X = dfPreparado.drop(columns=['diabetes'])
y = dfPreparado['diabetes']

# Confirmación de las dimensiones finales listas para Naive Bayes
print("Dimensiones de X (Características):", X.shape)
print("Dimensiones de y (Objetivo):", y.shape)

# Inspección de las primeras filas de las características (X)
X.head(6)

Dimensiones de X (Características): (6, 4)
Dimensiones de y (Objetivo): (6,)


,sexo,ciudad,colesterol,edad
0,1,1,0,0
1,2,5,2,2
2,2,2,1,1
3,1,3,2,2
4,2,0,1,1
5,1,4,3,2


<div id="model" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>3. Fase de Modelado</h2> </div>

In [ ]:

# Instanciación del clasificador Naive Bayes Categórico
# Se define el parámetro de suavizado de Laplace mediante alpha = 1.0 para evitar probabilidades cero
modelo_nb = CategoricalNB(alpha=1.0)

# Verificación de la estructura del objeto creado
print("Estado del modelo:")
print(modelo_nb)

Estado del modelo:
CategoricalNB()


In [21]:
# Ajuste del algoritmo utilizando el conjunto de características (X) y las etiquetas (y)
modelo_nb.fit(X, y)

print("Proceso de entrenamiento finalizado.")

Proceso de entrenamiento finalizado.


In [22]:
# Visualización de los elementos internos calculados durante el entrenamiento
print("Clases registradas en la variable objetivo:", modelo_nb.classes_)
print("Conteo de registros por clase (Frecuencia absoluta):", modelo_nb.class_count_)
print("Logaritmo de la probabilidad a priori de las clases (class_log_prior_):", modelo_nb.class_log_prior_)

Clases registradas en la variable objetivo: [0 1]
Conteo de registros por clase (Frecuencia absoluta): [3. 3.]
Logaritmo de la probabilidad a priori de las clases (class_log_prior_): [-0.69314718 -0.69314718]


<div id="predict" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>4. Fase de Predicción de Nuevos Samples</h2> </div>

In [ ]:

# Definición de dos nuevos pacientes con datos categóricos y continuos crudos
nuevos_datos_raw = pd.DataFrame([
    {'sexo': 1, 'ciudad': 'Quito', 'colesterol': 'medio', 'edad': 25},
    {'sexo': 2, 'ciudad': 'Cuenca', 'colesterol': 'muy alto', 'edad': 55}
])

print("Nuevos registros a evaluar (Datos crudos):")
nuevos_datos_raw.head()

Nuevos registros a evaluar (Datos crudos):


,sexo,ciudad,colesterol,edad
0,1,Quito,medio,25
1,2,Cuenca,muy alto,55


In [25]:
# Duplicar estructura para aplicar transformaciones
nuevos_datos_prep = nuevos_datos_raw.copy()

# 1. Mapeo de la variable nominal 'ciudad' con el codificador previamente ajustado
nuevos_datos_prep['ciudad'] = encoder_ciudad.transform(nuevos_datos_raw['ciudad'])

# 2. Mapeo jerárquico de la variable ordinal 'colesterol'
nuevos_datos_prep['colesterol'] = nuevos_datos_raw['colesterol'].map(mapeo_colesterol)

# 3. Discretización de la variable continua 'edad' aplicando los mismos intervalos [0, 30, 50, 100]
nuevos_datos_prep['edad'] = pd.cut(nuevos_datos_raw['edad'], bins=[0, 30, 50, 100], labels=[0, 1, 2]).astype(int)

print("Matriz de características de nuevos registros (X_nuevos):")
nuevos_datos_prep.head()

Matriz de características de nuevos registros (X_nuevos):


,sexo,ciudad,colesterol,edad
0,1,5,1,0
1,2,1,3,2


In [26]:
# Ejecución del método predict para obtener etiquetas numéricas
predicciones_numericas = modelo_nb.predict(nuevos_datos_prep)

# Decodificación inversa de los resultados mediante el codificador de la variable objetivo
predicciones_etiquetas = encoder_diabetes.inverse_transform(predicciones_numericas)

# Consolidación de resultados con los datos originales de entrada
resultados_clasificacion = nuevos_datos_raw.copy()
resultados_clasificacion['Clase_Asignada_Num'] = predicciones_numericas
resultados_clasificacion['Prediccion_Diabetes'] = predicciones_etiquetas

print("Predicciones finales generadas por el modelo:")
resultados_clasificacion.head()

Predicciones finales generadas por el modelo:


,sexo,ciudad,colesterol,edad,Clase_Asignada_Num,Prediccion_Diabetes
0,1,Quito,medio,25,0,no
1,2,Cuenca,muy alto,55,1,si


In [27]:
# Cálculo de la distribución de probabilidad para cada registro por clase
probabilidades = modelo_nb.predict_proba(nuevos_datos_prep)

# Anexión de las probabilidades calculadas (Clase 0: 'no', Clase 1: 'si')
resultados_clasificacion['Probabilidad_No'] = probabilidades[:, 0]
resultados_clasificacion['Probabilidad_Si'] = probabilidades[:, 1]

print("Distribución de probabilidad posterior por registro:")
resultados_clasificacion[['Prediccion_Diabetes', 'Probabilidad_No', 'Probabilidad_Si']].head()

Distribución de probabilidad posterior por registro:


,Prediccion_Diabetes,Probabilidad_No,Probabilidad_Si
0,no,0.666667,0.333333
1,si,0.272727,0.727273


<div id="conclusiones" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Conclusiones</h2> </div>

- Naturaleza del modelado (probabilístico vs. geométrico): A diferencia de KNN, que clasifica a los pacientes calculando distancias geométricas directas en un espacio multidimensional, Naive Bayes opera bajo un enfoque completamente probabilístico. Mientras que KNN requiere comparar cada nuevo registro con todo el conjunto de entrenamiento para hallar los vecinos más cercanos, Naive Bayes simplifica este proceso construyendo tablas de frecuencias y calculando verosimilitudes, lo que optimiza significativamente la carga computacional durante la fase de predicción.
- El supuesto de independencia frente a las variables clínicas: Naive Bayes se fundamenta en la suposición de que todas las variables predictoras (como edad, ciudad o colesterol) son condicionalmente independientes entre sí dada la clase. Aunque en la realidad biomédica factores como la edad y el colesterol suelen estar correlacionados, el algoritmo demostró ser sumamente robustas, ofreciendo clasificaciones consistentes a pesar de este sesgo teórico, a diferencia de KNN que se vuelve muy sensible si las variables no están correctamente escaladas o presentan redundancias.
- Sensibilidad a la escala y preprocesamiento: En la práctica previa con KNN, la estandarización mediante StandardScaler fue un paso obligatorio y crítico para evitar que variables con rangos numéricos amplios (como edad) distorsionaran el cálculo de la distancia euclídea y eclipsaran a los indicadores binarios. En contraste, para Naive Bayes la escala métrica pierde relevancia, requiriendo en su lugar una fase de discretización (como transformar la edad en rangos ordinales) para que el clasificador categórico pueda contabilizar adecuadamente las frecuencias empíricas de cada intervalo.
- Gestión de datos limitados y probabilidades cero: La incorporación del suavizado de Laplace ($\alpha = 1.0$) resultó determinante en Naive Bayes para manejar la escasez de datos del set local, evitando que combinaciones de síntomas no registradas previamente anularan por completo el cálculo matemático de la probabilidad posterior. En KNN, las muestras limitadas se traducen en una alta sensibilidad al ruido o a la elección del parámetro $k$, evidenciando que Naive Bayes tolera de mejor manera la ausencia de ciertas combinaciones de atributos en entornos clínicos pequeños.
- Interpretabilidad del diagnóstico y salida informativa: Mientras que KNN entrega un diagnóstico basado puramente en la votación mayoritaria de los vecinos más cercanos (una decisión más rígida o geométrica), Naive Bayes proporciona a través de predict_proba un desglose detallado de la probabilidad posterior para cada clase (si o no). Esto dota al modelo de una ventaja cualitativa en el entorno clínico, ya que no solo automatiza la clasificación del paciente, sino que cuantifica el nivel de certeza o incertidumbre asociado a la predicción, facilitando un criterio médico más flexible y matizado.

<div id="referencias" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Referencias</h2> </div>

Hastie, T., Tibshirani, R., & Friedman, J. (2009). The elements of statistical learning: Data mining, inference, and prediction (2a ed.). Springer.

James, G., Witten, D., Hastie, T., & Tibshirani, R. (2013). An introduction to statistical learning: with applications in R. Springer.

Kuhn, M., & Johnson, K. (2013). Applied predictive modeling. Springer.